# 📞 CallMeMaybe — Identificación de Operadores Ineficaces
## Proyecto Final: Telecomunicaciones

---

> **Google Drive (PDF + notebook):** `[pega aquí tu enlace de Drive]`

---

| Campo | Detalle |
|---|---|
| **Dataset principal** | `telecom_dataset_us.csv` — 53,902 registros |
| **Dataset clientes** | `telecom_clients_us.csv` — 732 registros |
| **Período** | Agosto – Noviembre 2019 |
| **Objetivo** | Identificar operadores ineficaces y validar hipótesis estadísticas |

---

### Contenido del notebook

1. Importaciones y carga de datos  
2. Análisis Exploratorio de Datos (EDA)  
3. Construcción de métricas por operador  
4. Identificación de operadores ineficaces  
5. Visualizaciones (dashboards)  
6. Pruebas de hipótesis estadísticas  
7. Conclusiones y recomendaciones


## 1. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings('ignore')

# Estilo global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

# Paleta de colores
C = dict(blue='#2563EB', red='#EF4444', green='#10B981',
         amber='#F59E0B', gray='#6B7280', dark='#1E3A5F', light='#EFF6FF')

print("✅ Librerías cargadas correctamente.")


## 2. Carga y limpieza de datos

### 2.1 Carga de datasets

In [ ]:
df  = pd.read_csv('telecom_dataset_us.csv')
dfc = pd.read_csv('telecom_clients_us.csv')

print(f"Dataset principal : {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Dataset clientes  : {dfc.shape[0]:,} filas × {dfc.shape[1]} columnas")
df.head()


### 2.2 Información general y tipos de datos

In [ ]:
print("=== telecom_dataset_us ===")
df.info()
print()
print("=== telecom_clients_us ===")
dfc.info()


### 2.3 Valores nulos

In [ ]:
print("Valores nulos — dataset principal:")
print(df.isnull().sum().to_string())
print()
print("Valores nulos — clientes:")
print(dfc.isnull().sum().to_string())


**Observaciones:**
- `operator_id` tiene **8,172 valores nulos (15.2%)** → registros del sistema sin operador asignado. Se excluirán del análisis por operador.
- `internal` tiene 117 nulos (0.2%) → se imputarán como `False` (externa).
- El resto de columnas están completas.

### 2.4 Limpieza y nuevas variables

In [ ]:
# Convertir fecha
df['date'] = pd.to_datetime(df['date'], utc=True)

# Imputar internal
df['internal'] = df['internal'].fillna(False).astype(bool)

# Calcular tiempo de espera (diferencia entre total y duración pura)
df['wait_time'] = df['total_call_duration'] - df['call_duration']

# Dataset solo con operador asignado
df_op = df.dropna(subset=['operator_id']).copy()
df_op['operator_id'] = df_op['operator_id'].astype(int)

print(f"Registros con operador asignado: {len(df_op):,} / {len(df):,}")
print(f"Operadores únicos  : {df_op['operator_id'].nunique():,}")
print(f"Usuarios únicos    : {df_op['user_id'].nunique():,}")
print(f"Rango de fechas    : {df_op['date'].min().date()} → {df_op['date'].max().date()}")


## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Estadísticas descriptivas

In [ ]:
df_op[['calls_count', 'call_duration', 'total_call_duration', 'wait_time']].describe().round(2)


### 3.2 Distribución de variables categóricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Dirección
dir_counts = df_op['direction'].value_counts()
axes[0].bar(dir_counts.index, dir_counts.values,
            color=[C['blue'], C['amber']], edgecolor='white', linewidth=0.8)
axes[0].set_title('Dirección de llamadas', fontweight='bold', color=C['dark'])
axes[0].set_ylabel('Registros')
for i, v in enumerate(dir_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df_op)*100:.1f}%)',
                 ha='center', fontsize=9, color=C['dark'])

# Interna vs externa
int_counts = df_op['internal'].map({True: 'Interna', False: 'Externa'}).value_counts()
axes[1].bar(int_counts.index, int_counts.values,
            color=[C['green'], C['blue']], edgecolor='white', linewidth=0.8)
axes[1].set_title('Tipo de llamada', fontweight='bold', color=C['dark'])
for i, v in enumerate(int_counts.values):
    axes[1].text(i, v + 200, f'{v:,}\n({v/len(df_op)*100:.1f}%)',
                 ha='center', fontsize=9, color=C['dark'])

# Llamadas perdidas
miss_counts = df_op['is_missed_call'].map({True: 'Perdida', False: 'Contestada'}).value_counts()
colors_miss = [C['red'], C['green']]
axes[2].bar(miss_counts.index, miss_counts.values,
            color=colors_miss, edgecolor='white', linewidth=0.8)
axes[2].set_title('Estado de llamada', fontweight='bold', color=C['dark'])
for i, v in enumerate(miss_counts.values):
    axes[2].text(i, v + 200, f'{v:,}\n({v/len(df_op)*100:.1f}%)',
                 ha='center', fontsize=9, color=C['dark'])

plt.suptitle('Distribución de variables categóricas', fontsize=13,
             fontweight='bold', color=C['dark'], y=1.02)
plt.tight_layout()
plt.show()


> ⚠️ **Hallazgo clave:** El **43.7% de las llamadas son perdidas**, una cifra muy elevada que sugiere problemas tanto individuales (operadores) como sistémicos (capacidad, enrutamiento).

### 3.3 Dashboard 1 — Duración de llamada + Composición interna/externa + Filtro por dirección

In [ ]:
# ── DASHBOARD 1 (Sugerencia 1 del enunciado) ──────────────────────────
for direction_filter in ['Todos', 'out', 'in']:
    if direction_filter == 'Todos':
        data_filt = df_op
    else:
        data_filt = df_op[df_op['direction'] == direction_filter]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    title_dir = {'Todos': 'Todas las llamadas', 'out': 'Solo salientes', 'in': 'Solo entrantes'}
    fig.suptitle(f'Dashboard 1 — Filtro: {title_dir[direction_filter]}',
                 fontsize=13, fontweight='bold', color=C['dark'])

    # Histograma duración
    dur_data = data_filt[data_filt['call_duration'] > 0]['call_duration'].clip(upper=5000)
    axes[0].hist(dur_data, bins=55, color=C['blue'], alpha=0.85,
                 edgecolor='white', linewidth=0.4)
    axes[0].axvline(dur_data.median(), color=C['red'], lw=2, ls='--',
                    label=f'Mediana: {dur_data.median():.0f}s')
    axes[0].axvline(dur_data.mean(), color=C['amber'], lw=2, ls='--',
                    label=f'Media: {dur_data.mean():.0f}s')
    axes[0].set_xlabel('Duración de llamada (segundos)')
    axes[0].set_ylabel('Frecuencia')
    axes[0].set_title('Histograma de duración de llamada', fontweight='bold')
    axes[0].legend(fontsize=9)

    # Pie interno vs externo
    int_c = data_filt['internal'].value_counts()
    vals  = [int_c.get(False, 0), int_c.get(True, 0)]
    axes[1].pie(vals, labels=['Externa', 'Interna'], autopct='%1.1f%%',
                colors=[C['blue'], C['green']], startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    axes[1].set_title('Participación interna vs externa', fontweight='bold')

    plt.tight_layout()
    plt.show()
    print(f"  Registros en filtro '{direction_filter}': {len(data_filt):,}\n")


### 3.4 Dashboard 2 — Llamadas por día + Composición interna/externa + Filtro por tipo

In [ ]:
# ── DASHBOARD 2 (Sugerencia 2 del enunciado) ──────────────────────────
for type_filter in ['Todos', True, False]:  # True=interna, False=externa
    if type_filter == 'Todos':
        data_filt = df_op
        label = 'Todos'
    else:
        data_filt = df_op[df_op['internal'] == type_filter]
        label = 'Internas' if type_filter else 'Externas'

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle(f'Dashboard 2 — Filtro: {label}',
                 fontsize=13, fontweight='bold', color=C['dark'])

    # Histograma llamadas por día
    daily = data_filt.groupby(data_filt['date'].dt.date)['calls_count'].sum()
    axes[0].bar(pd.to_datetime(daily.index), daily.values,
                color=C['blue'], alpha=0.8, width=0.9)
    roll7 = daily.rolling(7, min_periods=1).mean()
    axes[0].plot(pd.to_datetime(daily.index), roll7.values,
                 color=C['red'], lw=2.5, label='Media móvil 7d')
    axes[0].set_xlabel('Fecha')
    axes[0].set_ylabel('Número de llamadas')
    axes[0].set_title('Llamadas por día', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].legend()

    # Pie interno vs externo
    int_c = data_filt['internal'].value_counts()
    vals  = [int_c.get(False, 0), int_c.get(True, 0)]
    axes[1].pie(vals, labels=['Externa', 'Interna'], autopct='%1.1f%%',
                colors=[C['blue'], C['green']], startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    axes[1].set_title('Participación interna vs externa', fontweight='bold')

    plt.tight_layout()
    plt.show()
    print(f"  Registros en filtro '{label}': {len(data_filt):,}\n")


### 3.5 Distribución del tiempo de espera (llamadas entrantes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

wait_data = df_op[(df_op['direction'] == 'in') & (df_op['wait_time'] > 0)]['wait_time']

# Histograma completo
axes[0].hist(wait_data.clip(upper=2000), bins=60, color=C['amber'],
             alpha=0.85, edgecolor='white')
axes[0].axvline(wait_data.median(), color=C['red'], lw=2, ls='--',
                label=f'Mediana: {wait_data.median():.0f}s')
axes[0].set_xlabel('Tiempo de espera (segundos)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución tiempo de espera (clip 2000s)', fontweight='bold')
axes[0].legend()

# Boxplot por dirección
wait_by_dir = [
    df_op[(df_op['direction']=='in') & (df_op['wait_time']>0)]['wait_time'].clip(upper=1500),
    df_op[(df_op['direction']=='out') & (df_op['wait_time']>0)]['wait_time'].clip(upper=1500),
]
bp = axes[1].boxplot(wait_by_dir, patch_artist=True,
                     labels=['Entrantes', 'Salientes'], widths=0.5,
                     medianprops={'color': 'white', 'linewidth': 2.5})
bp['boxes'][0].set_facecolor(C['blue']); bp['boxes'][0].set_alpha(0.8)
bp['boxes'][1].set_facecolor(C['amber']); bp['boxes'][1].set_alpha(0.8)
axes[1].set_ylabel('Tiempo de espera (s)')
axes[1].set_title('Tiempo de espera por dirección', fontweight='bold')

plt.suptitle('Análisis del tiempo de espera', fontsize=13,
             fontweight='bold', color=C['dark'])
plt.tight_layout()
plt.show()

print(f"Tiempo de espera entrante — media: {wait_data.mean():.0f}s | "
      f"mediana: {wait_data.median():.0f}s | max: {wait_data.max():.0f}s")


## 4. Construcción de métricas por operador

Para medir la ineficiencia se calculan **tres métricas** basadas en los criterios de negocio:

| Métrica | Descripción | Indicador de ineficiencia |
|---|---|---|
| `missed_rate_in` | % llamadas entrantes perdidas (internas + externas) | Alto |
| `avg_wait_in` | Tiempo de espera promedio en llamadas entrantes | Alto |
| `out_calls` | Total de llamadas salientes realizadas | Bajo (si tiene salientes asignadas) |


In [ ]:
in_df  = df_op[df_op['direction'] == 'in']
out_df = df_op[df_op['direction'] == 'out']

# ── Tasa de llamadas entrantes perdidas ──────────────────────────────
missed_in = (
    in_df.groupby('operator_id')
    .apply(lambda x: x.loc[x['is_missed_call'], 'calls_count'].sum() / x['calls_count'].sum()
           if x['calls_count'].sum() > 0 else 0)
    .reset_index(name='missed_rate_in')
)

# ── Tiempo de espera promedio entrantes ──────────────────────────────
wait_in = (
    in_df.groupby('operator_id')['wait_time']
    .mean()
    .reset_index(name='avg_wait_in')
)

# ── Llamadas salientes totales ───────────────────────────────────────
out_calls = (
    out_df.groupby('operator_id')['calls_count']
    .sum()
    .reset_index(name='out_calls')
)

# ── Resumen general por operador ─────────────────────────────────────
op_base = df_op.groupby('operator_id').agg(
    total_calls   = ('calls_count', 'sum'),
    total_records = ('operator_id', 'count'),
    user_id       = ('user_id', 'first'),
).reset_index()

# ── Unir todo ────────────────────────────────────────────────────────
ops_with_out = out_df['operator_id'].unique()

op_metrics = (
    op_base
    .merge(missed_in, on='operator_id', how='left')
    .merge(wait_in,   on='operator_id', how='left')
    .merge(out_calls, on='operator_id', how='left')
)
op_metrics['missed_rate_in'] = op_metrics['missed_rate_in'].fillna(0)
op_metrics['avg_wait_in']    = op_metrics['avg_wait_in'].fillna(0)
op_metrics['out_calls']      = op_metrics['out_calls'].fillna(0)
op_metrics['has_outbound']   = op_metrics['operator_id'].isin(ops_with_out)

print(f"Operadores con métricas calculadas: {len(op_metrics):,}")
print(f"Operadores con salientes asignadas: {op_metrics['has_outbound'].sum():,}")
op_metrics[['operator_id','missed_rate_in','avg_wait_in','out_calls','has_outbound']].head(10)


### 4.1 Distribución de las tres métricas clave

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Tasa de pérdida
axes[0].hist(op_metrics['missed_rate_in'], bins=40, color=C['red'],
             alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Tasa de llamadas perdidas entrantes')
axes[0].set_ylabel('Nº operadores')
axes[0].set_title('Distribución tasa de pérdida', fontweight='bold')
axes[0].axvline(op_metrics['missed_rate_in'].median(), color=C['dark'],
                lw=2, ls='--', label=f"Mediana: {op_metrics['missed_rate_in'].median():.2f}")
axes[0].legend(fontsize=8)

# Tiempo de espera
axes[1].hist(op_metrics['avg_wait_in'].clip(upper=400), bins=40, color=C['amber'],
             alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Tiempo de espera promedio (s)')
axes[1].set_ylabel('Nº operadores')
axes[1].set_title('Distribución tiempo de espera\n(clip 400s)', fontweight='bold')
axes[1].axvline(op_metrics['avg_wait_in'].median(), color=C['dark'],
                lw=2, ls='--', label=f"Mediana: {op_metrics['avg_wait_in'].median():.0f}s")
axes[1].legend(fontsize=8)

# Llamadas salientes (operadores con outbound)
out_op = op_metrics[op_metrics['has_outbound']]['out_calls'].clip(upper=5000)
axes[2].hist(out_op, bins=40, color=C['blue'], alpha=0.8, edgecolor='white')
axes[2].set_xlabel('Total llamadas salientes')
axes[2].set_ylabel('Nº operadores')
axes[2].set_title('Distribución llamadas salientes\n(operadores con outbound, clip 5000)', fontweight='bold')
axes[2].axvline(out_op.median(), color=C['dark'],
                lw=2, ls='--', label=f"Mediana: {out_op.median():.0f}")
axes[2].legend(fontsize=8)

plt.suptitle('Distribución de métricas por operador', fontsize=13,
             fontweight='bold', color=C['dark'])
plt.tight_layout()
plt.show()


## 5. Identificación de operadores ineficaces

### 5.1 Score compuesto de ineficiencia

Se construye un **score ponderado** normalizando cada métrica al rango [0, 1] con Min-Max Scaling:

$$\text{Score} = 0.40 \times s_{\text{missed}} + 0.35 \times s_{\text{wait}} + 0.25 \times s_{\text{out\_inv}}$$

- $s_{\text{missed}}$: tasa de pérdida normalizada (mayor = peor)
- $s_{\text{wait}}$: tiempo de espera normalizado (mayor = peor)  
- $s_{\text{out\_inv}}$: salientes normalizadas e **invertidas** (menos salientes = peor)

El umbral de ineficiencia se fija en el **percentil 75** del score.


In [ ]:
scaler = MinMaxScaler()

# Normalizar missed_rate y wait
op_metrics[['s_missed', 's_wait']] = scaler.fit_transform(
    op_metrics[['missed_rate_in', 'avg_wait_in']]
)

# Invertir salientes (solo para operadores con outbound)
op_metrics['s_out'] = 0.0
mask = op_metrics['has_outbound']
if mask.sum() > 0:
    scaled_out = scaler.fit_transform(op_metrics.loc[mask, ['out_calls']])
    op_metrics.loc[mask, 's_out'] = 1 - scaled_out[:, 0]

# Score compuesto
op_metrics['inefficiency_score'] = (
    0.40 * op_metrics['s_missed'] +
    0.35 * op_metrics['s_wait']   +
    0.25 * op_metrics['s_out']
)

# Clasificación por percentil 75
threshold = op_metrics['inefficiency_score'].quantile(0.75)
op_metrics['is_inefficient'] = op_metrics['inefficiency_score'] >= threshold

n_inef = op_metrics['is_inefficient'].sum()
n_ef   = (~op_metrics['is_inefficient']).sum()

print(f"Umbral de ineficiencia (P75) : {threshold:.4f}")
print(f"Operadores ineficaces        : {n_inef:,} ({n_inef/len(op_metrics)*100:.1f}%)")
print(f"Operadores eficaces          : {n_ef:,} ({n_ef/len(op_metrics)*100:.1f}%)")


### 5.2 Distribución del score e identificación del umbral

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Histograma por grupo
eff_sc   = op_metrics[~op_metrics['is_inefficient']]['inefficiency_score']
ineff_sc = op_metrics[ op_metrics['is_inefficient']]['inefficiency_score']

axes[0].hist(eff_sc,   bins=40, color=C['green'], alpha=0.75,
             label=f'Eficaces ({len(eff_sc)})',   edgecolor='white')
axes[0].hist(ineff_sc, bins=40, color=C['red'],   alpha=0.75,
             label=f'Ineficaces ({len(ineff_sc)})', edgecolor='white')
axes[0].axvline(threshold, color=C['dark'], lw=2.5, ls='--',
                label=f'Umbral P75: {threshold:.3f}')
axes[0].set_xlabel('Score de ineficiencia')
axes[0].set_ylabel('Nº operadores')
axes[0].set_title('Distribución del score por grupo', fontweight='bold', color=C['dark'])
axes[0].legend()

# Pie eficaces vs ineficaces
axes[1].pie([n_ef, n_inef],
            labels=[f'Eficaces\n{n_ef}', f'Ineficaces\n{n_inef}'],
            autopct='%1.1f%%', colors=[C['green'], C['red']],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Clasificación de operadores', fontweight='bold', color=C['dark'])

plt.suptitle('Identificación de operadores ineficaces', fontsize=13,
             fontweight='bold', color=C['dark'])
plt.tight_layout()
plt.show()


### 5.3 Top 20 operadores más ineficaces

In [ ]:
top20 = op_metrics.sort_values('inefficiency_score', ascending=False).head(20)

display_cols = ['operator_id', 'missed_rate_in', 'avg_wait_in',
                'out_calls', 'has_outbound', 'inefficiency_score']

styled = (
    top20[display_cols]
    .style
    .format({
        'missed_rate_in': '{:.1%}',
        'avg_wait_in': '{:.1f}s',
        'out_calls': '{:,.0f}',
        'inefficiency_score': '{:.4f}',
    })
    .background_gradient(subset=['inefficiency_score'], cmap='RdYlGn_r')
    .set_caption('Top 20 operadores con mayor score de ineficiencia')
)
display(styled)


### 5.4 Ranking visual: Top 15 operadores más ineficaces

In [ ]:
top15 = op_metrics.sort_values('inefficiency_score', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = [C['red'] if s >= threshold else C['amber']
              for s in top15['inefficiency_score']]
bars = ax.barh(top15['operator_id'].astype(str),
               top15['inefficiency_score'],
               color=colors_bar, edgecolor='white', linewidth=0.5)
ax.axvline(threshold, color=C['dark'], lw=2, ls='--',
           label=f'Umbral P75: {threshold:.3f}')
ax.set_xlabel('Score de ineficiencia', fontsize=11)
ax.set_title('Top 15 operadores con mayor score de ineficiencia',
             fontsize=13, fontweight='bold', color=C['dark'])
ax.legend()

for bar, val in zip(bars, top15['inefficiency_score']):
    ax.text(bar.get_width() + 0.004,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8.5, color=C['dark'])

plt.tight_layout()
plt.show()


### 5.5 Comparativa de métricas: eficaces vs ineficaces

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

groups  = [op_metrics[~op_metrics['is_inefficient']],
           op_metrics[ op_metrics['is_inefficient']]]
glabels = ['Eficaces', 'Ineficaces']
gcolors = [C['green'], C['red']]
metrics = [
    ('missed_rate_in', 'Tasa de llamadas\nentrantes perdidas (%)'),
    ('avg_wait_in',    'Tiempo de espera\npromedio (s)'),
    ('out_calls',      'Llamadas salientes\ntotales'),
]

for ax, (col, ylabel) in zip(axes, metrics):
    data = [g[col].clip(upper=g[col].quantile(0.95)).values for g in groups]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops={'color': 'white', 'linewidth': 2.5},
                    flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4})
    for patch, color in zip(bp['boxes'], gcolors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(glabels)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, fontweight='bold', color=C['dark'], fontsize=10)

plt.suptitle('Comparativa de métricas: eficaces vs ineficaces\n(valores hasta P95)',
             fontsize=12, fontweight='bold', color=C['dark'])
plt.tight_layout()
plt.show()


### 5.6 Mapa de correlaciones

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

corr_data = op_metrics[['missed_rate_in', 'avg_wait_in', 'out_calls', 'inefficiency_score']].copy()
corr_data.columns = ['% Perdidas', 'T. Espera (s)', 'Salientes', 'Score Ineficiencia']

sns.heatmap(corr_data.corr(), annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            annot_kws={'size': 11, 'weight': 'bold'})
ax.set_title('Correlación entre métricas de operadores',
             fontsize=13, fontweight='bold', color=C['dark'])
plt.tight_layout()
plt.show()


## 6. Pruebas de hipótesis estadísticas

Se utiliza la **prueba de Mann-Whitney U** (no paramétrica) ya que las distribuciones son asimétricas y no siguen normalidad.  
Nivel de significancia: **α = 0.05**

---

### Hipótesis planteadas

| # | H₀ (hipótesis nula) | H₁ (hipótesis alternativa) |
|---|---|---|
| H1 | Los ineficaces tienen la misma tasa de pérdida que los eficaces | Los ineficaces tienen **mayor** tasa de pérdida |
| H2 | Los ineficaces tienen el mismo tiempo de espera que los eficaces | Los ineficaces tienen **mayor** tiempo de espera |
| H3 | Los eficaces realizan las mismas salientes que los ineficaces | Los eficaces realizan **más** llamadas salientes |


In [ ]:
alpha = 0.05
eff   = op_metrics[~op_metrics['is_inefficient']]
ineff = op_metrics[ op_metrics['is_inefficient']]

# H1: Tasa de llamadas perdidas
u1, p1 = stats.mannwhitneyu(ineff['missed_rate_in'], eff['missed_rate_in'],
                             alternative='greater')

# H2: Tiempo de espera entrante
u2, p2 = stats.mannwhitneyu(ineff['avg_wait_in'], eff['avg_wait_in'],
                             alternative='greater')

# H3: Llamadas salientes (solo operadores con outbound asignado)
eff_out   = eff[eff['has_outbound']]
ineff_out = ineff[ineff['has_outbound']]
u3, p3 = stats.mannwhitneyu(eff_out['out_calls'], ineff_out['out_calls'],
                             alternative='greater')

results = {
    'H1 — Tasa de pérdida': (u1, p1),
    'H2 — Tiempo de espera': (u2, p2),
    'H3 — Llamadas salientes': (u3, p3),
}

print(f"{'Hipótesis':<30} {'U-stat':>12} {'p-value':>14} {'Decisión':>20}")
print("─" * 80)
for name, (u, p) in results.items():
    decision = '✓ RECHAZAR H₀' if p < alpha else '✗ No rechazar H₀'
    print(f"{name:<30} {u:>12,.1f} {p:>14.2e} {decision:>20}")


### 6.1 Visualización de resultados de hipótesis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
hyp_data = [
    ('missed_rate_in',  'Tasa de pérdida',     p1, 'H1'),
    ('avg_wait_in',     'Tiempo de espera (s)', p2, 'H2'),
    ('out_calls',       'Llamadas salientes',   p3, 'H3'),
]
groups_h = [
    (eff['missed_rate_in'],   ineff['missed_rate_in']),
    (eff['avg_wait_in'].clip(upper=300), ineff['avg_wait_in'].clip(upper=300)),
    (eff_out['out_calls'].clip(upper=3000), ineff_out['out_calls'].clip(upper=3000)),
]

for ax, (col, ylabel, p, hname), (g_eff, g_ineff) in zip(axes, hyp_data, groups_h):
    bp = ax.boxplot([g_eff.values, g_ineff.values],
                    patch_artist=True, widths=0.5,
                    medianprops={'color': 'white', 'linewidth': 2.5})
    bp['boxes'][0].set_facecolor(C['green']); bp['boxes'][0].set_alpha(0.8)
    bp['boxes'][1].set_facecolor(C['red']);   bp['boxes'][1].set_alpha(0.8)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Eficaces', 'Ineficaces'])
    ax.set_ylabel(ylabel)
    verdict = '✓ Rechaza H₀' if p < alpha else '✗ No rechaza H₀'
    color_v = C['green'] if p < alpha else C['red']
    ax.set_title(f'{hname}: {ylabel}\np = {p:.2e} — {verdict}',
                 fontweight='bold', color=color_v, fontsize=9.5)

plt.suptitle('Pruebas de hipótesis: comparativa por grupo',
             fontsize=13, fontweight='bold', color=C['dark'])
plt.tight_layout()
plt.show()


## 7. Conclusiones y recomendaciones

### 7.1 Hallazgos principales

#### EDA
- El dataset contiene **53,902 registros** (ago–nov 2019) con **1,092 operadores** y **290 clientes**.
- El **43.7% de las llamadas son perdidas**, cifra sistémicamente elevada que requiere atención tanto a nivel de operador individual como de capacidad global.
- La distribución de duración y tiempo de espera es **muy asimétrica**: medianas muy inferiores a las medias, con valores extremos de hasta 166,000 segundos.
- Las llamadas predominan en dirección **saliente (59.1%)** y son mayoritariamente **externas (86.5%)**.

#### Operadores ineficaces
- Se identificaron **273 operadores ineficaces** (25% superior según score compuesto).
- Los 3 operadores más críticos presentan: 100% de tasa de pérdida (op. 948756), tiempos de espera de 853s y 1,039s (ops. 906070 y 919794).
- La **tasa de llamadas perdidas** y el **tiempo de espera** son los mejores predictores de ineficiencia.

#### Hipótesis
| Hipótesis | Resultado | Interpretación |
|---|---|---|
| H1: ineficaces → más llamadas perdidas | ✅ **Se rechaza H₀** (p ≈ 0) | Diferencia estadísticamente significativa |
| H2: ineficaces → mayor tiempo de espera | ✅ **Se rechaza H₀** (p ≈ 0) | Diferencia estadísticamente significativa |
| H3: eficaces → más llamadas salientes | ❌ **No se rechaza H₀** (p = 0.999) | No hay evidencia de diferencia |

> **H3:** El volumen de salientes por sí solo no distingue eficiencia. Se recomienda incorporar la **tasa de éxito de salientes** como métrica complementaria.

### 7.2 Recomendaciones

1. **Intervención inmediata** en los 20 operadores con score > 0.35.
2. **Revisar enrutamiento** para operadores con tiempo de espera > 600s.
3. **Establecer alertas automáticas**: tasa de pérdida > 30% o espera > 120s.
4. **Mejorar la métrica de salientes**: usar tasa de éxito, no solo volumen.
5. **Investigar causas sistémicas** del 43.7% de llamadas perdidas globales.
6. **Ampliar el período** de análisis para confirmar si la ineficiencia es crónica o puntual.

---

*Notebook generado para el Proyecto Final de Telecomunicaciones — CallMeMaybe*


## 8. Fuentes y referencias

1. **Pandas Documentation** — https://pandas.pydata.org/docs/  
   *Agrupaciones (groupby, agg), merge de datasets, manejo de nulos y fechas.*

2. **SciPy Stats — Mann-Whitney U** — https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html  
   *Implementación de la prueba no paramétrica para comparar grupos sin asumir normalidad.*

3. **Scikit-learn — MinMaxScaler** — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html  
   *Normalización de métricas heterogéneas para construir el score compuesto.*

4. **Seaborn Documentation** — https://seaborn.pydata.org/  
   *Boxplots comparativos entre grupos y mapas de calor de correlaciones.*

5. **Matplotlib Documentation** — https://matplotlib.org/stable/contents.html  
   *Histogramas, gráficos de barras y series temporales de volumen diario.*

6. **Towards Data Science — Composite Scoring for Operations** — https://towardsdatascience.com/  
   *Marco conceptual para scores compuestos con pesos y umbrales percentílicos.*

7. **Statistics How To — Mann-Whitney U Test** — https://www.statisticshowto.com/mann-whitney-u-test/  
   *Cuándo usar pruebas no paramétricas y cómo interpretar U y p-value.*

8. **Yandex Practicum — Proyecto Telecomunicaciones** — https://practicum.yandex.com/  
   *Contexto del negocio, descripción de datasets y criterios de ineficiencia para CallMeMaybe.*

9. **NumPy Documentation** — https://numpy.org/doc/  
   *Cálculos de percentiles para el umbral del score y operaciones vectoriales.*

10. **Kaggle — Call Center Analytics** — https://www.kaggle.com/  
    *Referencia de métricas estándar en análisis de centros de llamadas (AHT, abandon rate, etc.).*
